<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_02_03_hpc_parallel_processing_snow_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Data Processing and Analysis with {snow}


The **{snow}** package in R, which stands for "Simple Network of Workstations," is designed for parallel computing, allowing users to distribute computational tasks across multiple processors or computers to speed up processing. It provides a high-level interface for parallelizing R code, particularly for tasks that can be split into independent subtasks, such as simulations, bootstrapping, or large-scale data processing.

The {snow} package, developed by Luke Tierney and others, enables parallel computing in R by leveraging multiple processors or machines in a cluster. It abstracts much of the complexity of parallel programming, making it accessible for users who need to perform computationally intensive tasks. {snow} supports various parallel backends, including MPI (Message Passing Interface), PVM (Parallel Virtual Machine), and socket-based communication, allowing flexibility depending on the system setup.

The package is particularly useful for:

- **Embarrassingly parallel problems**: Tasks that can be divided into independent subtasks with no need for communication between them (e.g., Monte Carlo simulations, cross-validation).
- **Cluster computing**: Distributing tasks across multiple machines in a network.
- **Multi-core processing**: Utilizing multiple cores on a single machine.

{snow} is lightweight, flexible, and integrates well with R's functional programming paradigm. It is often used alongside other parallel computing packages like {parallel} or {foreach} for specific use cases.


## How {snow} Works

{snow} operates by creating a **cluster** of worker processes (or nodes) that execute tasks in parallel. The main steps in using {snow} are:

1. **Cluster Setup**:
   - Initialize a cluster using a chosen communication backend (e.g., MPI, sockets, or PVM).
   - Specify the number of nodes (workers) and their configuration (e.g., local cores or remote machines).
   - Common functions: `makeCluster()` to create the cluster, specifying the type (e.g., "MPI", "SOCK", "PVM") and number of nodes.

2. **Task Distribution**:
   - Divide the computational task into smaller, independent subtasks.
   - Use {snow} functions like `clusterApply()`, `clusterApplyLB()`, or `parLapply()` to distribute these tasks across the cluster's nodes.
   - Each node processes its assigned task independently, and results are collected back to the master process.

3. **Communication**:
   - {snow} handles communication between the master (the main R session) and worker nodes using the chosen backend.
   - Workers execute R functions or scripts sent by the master, and results are returned via the communication protocol (e.g., sockets or MPI).

4. **Cluster Termination**:
   - Once tasks are complete, the cluster is shut down using `stopCluster()` to free resources.


## Key Functions in {snow}

Here are the core functions in {snow} and their purposes:

- **`makeCluster(spec, type, ...)`**: Creates a cluster of workers. `spec` specifies the number of nodes or machine names, and `type` defines the backend (e.g., "SOCK" for sockets, "MPI" for MPI).
- **`clusterApply(cl, x, fun, ...)`**: Applies a function `fun` to each element of `x` across the cluster `cl`. Each worker processes a subset of `x`.
- **`clusterApplyLB(cl, x, fun, ...)`**: Similar to `clusterApply`, but with load balancing, dynamically assigning tasks to workers as they become available.
- **`parLapply(cl, x, fun, ...)`**: A parallel version of `lapply`, distributing tasks across the cluster.
- **`clusterExport(cl, list)`**: Exports variables or functions from the master's environment to the workers' environments.
- **`clusterEvalQ(cl, expr)`**: Evaluates an expression on each worker, useful for initializing worker environments (e.g., loading packages).
- **`stopCluster(cl)`**: Shuts down the cluster, freeing resources.


## Example Workflow

Here's a simple example of using {snow} for parallel computation with a socket-based cluster:

**Output** (assuming no errors): `[1]   1   4   9  16  25  36  49  64  81 100`

In this example:

- A cluster of 4 workers is created using sockets.
- The `my_data` vector is exported to all workers.
- The `square` function is applied to each element of `my_data` in parallel using `parLapply`.
- Results are collected and returned as a list, which is unlisted for display.


In [ ]:
%%R
library(snow)

# Create a cluster with 4 workers (using sockets)
cl <- makeCluster(4, type = "SOCK")

# Export a variable to all workers
my_data <- 1:10
clusterExport(cl, "my_data")

# Define a function to compute squares
square <- function(x) x^2

# Apply the function in parallel
result <- parLapply(cl, my_data, square)

# Stop the cluster
stopCluster(cl)

# View results
print(unlist(result))


## How It Works Internally

- **Master-Worker Model**: The master R session coordinates tasks, while workers (separate R processes) execute them. The master sends tasks and collects results.
- **Communication Backend**:
  - **SOCK**: Uses TCP/IP sockets for communication, suitable for local or networked machines.
  - **MPI**: Uses the MPI standard (via {Rmpi} package) for high-performance clusters.
  - **PVM**: Uses the Parallel Virtual Machine system (less common today).
- **Load Balancing**: Functions like `clusterApplyLB` ensure efficient task distribution by assigning new tasks to workers as they complete their current ones.
- **Serialization**: R objects (data, functions) are serialized and sent to workers, which deserialize and process them. Results are serialized and sent back to the master.


## Advantages of {snow}

- **Simplicity**: Provides a high-level interface, hiding much of the complexity of parallel programming.
- **Flexibility**: Supports multiple backends, making it adaptable to different hardware setups (local multi-core machines or distributed clusters).
- **Scalability**: Works for both small-scale (e.g., multi-core laptops) and large-scale (e.g., HPC clusters) computations.
- **Integration**: Works seamlessly with R's functional programming style and other packages like {foreach}.


## Limitations

- **Overhead**: Setting up a cluster and communicating data introduces overhead, which may outweigh benefits for small tasks.
- **Dependencies**: Some backends (e.g., MPI) require additional software installation and configuration (e.g., {Rmpi}).
- **Memory**: Each worker loads a copy of the data and R environment, which can be memory-intensive.
- **Not for Interdependent Tasks**: {snow} is best for embarrassingly parallel tasks; tasks requiring frequent communication between workers are better suited for other frameworks (e.g., MPI directly).


## Practical Considerations

- **Choosing a Backend**: Use "SOCK" for simplicity on local machines or small networks. Use "MPI" for high-performance clusters if {Rmpi} is installed.
- **Environment Setup**: Ensure all necessary packages and objects are exported to workers using `clusterExport` or `clusterEvalQ`.
- **Task Granularity**: Break tasks into appropriately sized chunks to balance computation time and communication overhead.
- **Error Handling**: Use try-catch mechanisms in functions to handle errors on workers, as they run independently.


## Parallel Data Processing and Basic Statistics with {snow}

This tutorial demonstrates how to use the {snow} package in R for parallel computing, applied to data processing and basic statistical analyses on a large dataset: the NYC Yellow Taxi trip records for January 2023 (`yellow_tripdata_2023-01.parquet`). The dataset contains approximately 3 million rows of taxi trip data, including variables like pickup/dropoff times, distances, fares, and more. We'll cover parallel data loading/processing, descriptive statistics, correlation, and simple/multiple linear regression.

{snow} enables distributing tasks across multiple cores or machines, which is ideal for large datasets to reduce computation time. We'll focus on "embarrassingly parallel" tasks, such as processing data chunks independently and combining results.

**Note (Colab / `rpy2`):** Socket clusters (`SOCK`) spawn separate R processes. If `makeCluster` fails in a hosted environment, reduce workers or run the data sections locally.


## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab's Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there. Adjust `dataFolder` in the data-setup cell to point at your Parquet file (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Check and Install Required R Packages

The cells below mirror the Quarto tutorial: define packages, install to Drive (Colab), verify, load.


### Define required packages


In [ ]:
%%R
packages <- c(
  'tidyverse',
  'data.table',
  'feather',
  'arrow',
  'parallel',
  'future',
  'furrr',
  'future.apply',
  'nycflights13',
  'glue',
  'snow',
  'matrixStats',
  'reshape2'
)


### Install missing packages


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- pac


### Check loaded packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


### Loading and Exploring the Data Sequentially

Load the data using {arrow} for memory efficiency. Convert to a data.frame for easier manipulation (note: this collects the data into memory).

#### Set `dataFolder` for Colab (edit path)


In [ ]:
%%R
# Colab: set folder containing yellow_tripdata_2023-01.parquet
dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
dataFolder <- sub("/$", "", dataFolder)
dataFolder <- paste0(dataFolder, "/")


#### Local / repo path (edit if running outside this project)


In [ ]:
%%R
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
df <- read_parquet(paste0(dataFolder, "yellow_tripdata_2023-01.parquet"))
dim(df)


The dataset is large, so sequential processing can be slow for intensive tasks. We'll use {snow} to parallelize.

### Setting Up the {snow} Cluster

Create a cluster using available cores (e.g., 4 workers). Use the `"SOCK"` type for local multi-core processing. Always call `stopCluster(cl)` when finished to free resources (shown in cells below after each parallel block).


### Parallel Data Processing: Cleaning and Feature Engineering

For large data, split into chunks, process each in parallel (e.g., add features, filter invalid rows), and recombine.


In [ ]:
%%R
library(snow)
n_chunks <- parallel::detectCores() - 1
chunk_indices <- rep(1:n_chunks, each = ceiling(nrow(df) / n_chunks), length.out = nrow(df))
df_chunks <- split(df, chunk_indices)


In [ ]:
%%R
process_chunk <- function(chunk) {
  chunk$trip_duration <- as.numeric(difftime(chunk$tpep_dropoff_datetime,
                                             chunk$tpep_pickup_datetime, units = "mins"))
  chunk <- chunk[chunk$trip_distance > 0 &
                 chunk$fare_amount > 0 &
                 chunk$trip_duration > 0 &
                 chunk$passenger_count >= 1 &
                 chunk$passenger_count <= 6, ]
  return(chunk)
}


In [ ]:
%%R
n_cores <- parallel::detectCores() - 1
cl <- makeCluster(n_cores, type = "SOCK")
clusterEvalQ(cl, {
  library(arrow)
})
clusterExport(cl, "process_chunk")
processed_chunks <- parLapply(cl, df_chunks, process_chunk)
df_processed <- do.call(rbind, processed_chunks)
dim(df_processed)


In [ ]:
%%R
if (!is.null(cl)) {
  tryCatch({
    stopCluster(cl)
  }, error = function(e) {
    cat("Error stopping cluster:", conditionMessage(e), "\n")
  })
  cl <- NULL
}


This parallelizes the cleaning, speeding up for large datasets.


### Parallel Descriptive Statistics

For descriptive stats (mean, median, sd, min, max), compute on chunks and aggregate. Focus on numeric variables: trip_distance, fare_amount, tip_amount, trip_duration.


In [ ]:
%%R
library(snow)
n_cores <- parallel::detectCores() - 1
cl <- NULL
tryCatch({
  cl <- makeCluster(n_cores, type = "SOCK")
}, error = function(e) {
  cat("Error creating cluster:", conditionMessage(e), "\n")
  stop("Cluster creation failed.")
})
clusterEvalQ(cl, {
  library(arrow)
  library(matrixStats)
})
vars <- c("trip_distance", "fare_amount", "tip_amount", "trip_duration")
local_stats <- function(chunk, vars) {
  list(
    n = nrow(chunk),
    sums = colSums(chunk[, vars], na.rm = TRUE),
    sum_squares = colSums(chunk[, vars]^2, na.rm = TRUE),
    mins = apply(chunk[, vars], 2, min, na.rm = TRUE),
    maxs = apply(chunk[, vars], 2, max, na.rm = TRUE),
    medians = apply(chunk[, vars], 2, median, na.rm = TRUE)
  )
}
n_chunks <- n_cores
df_chunks <- split(df_processed, rep(1:n_chunks, each = ceiling(nrow(df_processed) / n_chunks), length.out = nrow(df_processed)))
clusterExport(cl, c("local_stats", "vars"))
local_results <- parLapply(cl, df_chunks, local_stats, vars = vars)
total_n <- sum(sapply(local_results, function(x) x$n))
global_sums <- rowSums(sapply(local_results, function(x) x$sums))
global_sum_squares <- rowSums(sapply(local_results, function(x) x$sum_squares))
global_means <- global_sums / total_n
global_vars <- (global_sum_squares / total_n) - (global_means)^2
global_sds <- sqrt(global_vars)
global_mins <- matrixStats::rowMins(sapply(local_results, function(x) x$mins), na.rm = TRUE)
global_maxs <- matrixStats::rowMaxs(sapply(local_results, function(x) x$maxs), na.rm = TRUE)
global_medians <- apply(df_processed[, vars], 2, median, na.rm = TRUE)
descriptive_stats <- data.frame(
  Variable = vars,
  Mean = global_means,
  SD = global_sds,
  Min = global_mins,
  Median = global_medians,
  Max = global_maxs
)
print(descriptive_stats)


In [ ]:
%%R
if (!is.null(cl)) {
  tryCatch({
    stopCluster(cl)
  }, error = function(e) {
    cat("Error stopping cluster:", conditionMessage(e), "\n")
  })
  cl <- NULL
}


This approach avoids loading the full dataset on each worker and computes aggregates efficiently.


### Parallel Correlation Analysis

For parallel correlation: compute local covariance matrices, then combine. Select numeric vars.


In [ ]:
%%R
library(snow)
n_cores <- parallel::detectCores() - 1
cl <- NULL
tryCatch({
  cl <- makeCluster(n_cores, type = "SOCK")
}, error = function(e) {
  cat("Error creating cluster:", conditionMessage(e), "\n")
  stop("Cluster creation failed.")
})
clusterEvalQ(cl, {
  library(arrow)
  library(matrixStats)
})
vars <- c("trip_distance", "fare_amount", "tip_amount", "trip_duration")
num_df <- df_processed[, vars, drop = FALSE]
if (!all(sapply(num_df, is.numeric))) {
  stop("Non-numeric columns detected in num_df. Please ensure all variables are numeric.")
}
num_df <- num_df[complete.cases(num_df) & apply(num_df, 1, function(x) all(is.finite(x))), ]
if (nrow(num_df) == 0) {
  stop("No valid rows remain after removing NAs and infinite values.")
}
local_cov <- function(chunk) {
  if (nrow(chunk) < 2) {
    return(list(n = 0, means = rep(0, length(vars)), sum_prods = matrix(0, nrow = length(vars), ncol = length(vars))))
  }
  chunk <- chunk[complete.cases(chunk) & apply(chunk, 1, function(x) all(is.finite(x))), , drop = FALSE]
  n <- nrow(chunk)
  if (n < 2) {
    return(list(n = 0, means = rep(0, length(vars)), sum_prods = matrix(0, nrow = length(vars), ncol = length(vars))))
  }
  means <- colMeans(chunk, na.rm = TRUE)
  sum_prods <- crossprod(scale(chunk, center = TRUE, scale = FALSE)) / (n - 1) * (n - 1)
  list(n = n, means = means, sum_prods = sum_prods)
}
n_chunks <- n_cores
num_chunks <- split(num_df, rep(1:n_chunks, each = ceiling(nrow(num_df) / n_chunks), length.out = nrow(num_df)))
clusterExport(cl, c("local_cov", "vars"))
local_covs <- parLapply(cl, num_chunks, local_cov)
cat("Local chunk sizes:\n")
print(sapply(local_covs, function(x) x$n))
total_n <- sum(sapply(local_covs, function(x) x$n))
if (total_n < 2) {
  stop("Not enough valid observations across chunks to compute correlations.")
}
global_means <- rowSums(sapply(local_covs, function(x) x$means * x$n)) / total_n
global_cov <- Reduce(`+`, lapply(local_covs, function(x) {
  if (x$n == 0) {
    matrix(0, nrow = length(vars), ncol = length(vars))
  } else {
    x$sum_prods + outer(x$means - global_means, x$means - global_means) * x$n
  }
})) / total_n
global_sd <- sqrt(diag(global_cov))
if (any(global_sd == 0)) {
  warning("Zero standard deviation detected for some variables. Correlation will be NA for those.")
}
global_cor <- global_cov / outer(global_sd, global_sd)
dimnames(global_cor) <- list(vars, vars)
print(round(global_cor, 3))


In [ ]:
%%R
if (!is.null(cl)) {
  tryCatch({
    stopCluster(cl)
  }, error = function(e) {
    cat("Error stopping cluster:", conditionMessage(e), "\n")
  })
  cl <- NULL
}


### Parallel Linear Regression


In [ ]:
%%R
library(snow)
n_cores <- parallel::detectCores() - 1
cl <- NULL
tryCatch({
  cl <- makeCluster(n_cores, type = "SOCK")
}, error = function(e) {
  cat("Error creating cluster:", conditionMessage(e), "\n")
  stop("Cluster creation failed.")
})
vars <- c("trip_distance", "fare_amount", "passenger_count", "trip_duration")
reg_df <- df_processed[, vars, drop = FALSE]
reg_df <- reg_df[complete.cases(reg_df) & apply(reg_df, 1, function(x) all(is.finite(x))), ]
fit_model <- function(data, formula) {
  if (nrow(data) < 2) {
    return(list(coefficients = NULL, n = 0))
  }
  model <- lm(formula, data = data)
  list(coefficients = coef(model), n = nrow(data))
}
n_chunks <- n_cores
chunks <- split(reg_df, rep(1:n_chunks, each = ceiling(nrow(reg_df) / n_chunks), length.out = nrow(reg_df)))


#### Simple Linear Regression (SLR)


In [ ]:
%%R
ols_formula <- fare_amount ~ trip_distance
clusterExport(cl, c("fit_model", "reg_df", "ols_formula"))
ols_results <- parLapply(cl, chunks, function(chunk) fit_model(chunk, ols_formula))
ols_n <- sapply(ols_results, function(x) x$n)
total_n <- sum(ols_n)
if (total_n < 2) stop("Not enough valid observations to fit model.")
ols_coefs <- Reduce("+", lapply(ols_results, function(x) if (is.null(x$coefficients)) rep(0, 2) else x$coefficients * x$n)) / total_n
cat("OLS Coefficients (Parallel):\n")
names(ols_coefs) <- c("(Intercept)", "trip_distance")
print(round(ols_coefs, 3))
ols_seq <- lm(fare_amount ~ trip_distance, data = reg_df)
cat("\nOLS Coefficients (Sequential):\n")
print(round(coef(ols_seq), 3))


#### Multiple Linear Regression (MLR)


In [ ]:
%%R
mlr_formula <- fare_amount ~ trip_distance + passenger_count + trip_duration
clusterExport(cl, c("fit_model", "reg_df", "mlr_formula"))
mlr_results <- parLapply(cl, chunks, function(chunk) fit_model(chunk, mlr_formula))
mlr_n <- sapply(mlr_results, function(x) x$n)
total_n <- sum(mlr_n)
if (total_n < 2) stop("Not enough valid observations to fit model.")
mlr_coefs <- Reduce("+", lapply(mlr_results, function(x) if (is.null(x$coefficients)) rep(0, 4) else x$coefficients * x$n)) / total_n
cat("\nMLR Coefficients (Parallel):\n")
names(mlr_coefs) <- c("(Intercept)", "trip_distance", "passenger_count", "trip_duration")
print(round(mlr_coefs, 3))
mlr_seq <- lm(fare_amount ~ trip_distance + passenger_count + trip_duration, data = reg_df)
cat("\nMLR Coefficients (Sequential):\n")
print(round(coef(mlr_seq), 3))


In [ ]:
%%R
if (!is.null(cl)) {
  tryCatch({
    stopCluster(cl)
  }, error = function(e) {
    cat("Error stopping cluster:", conditionMessage(e), "\n")
  })
  cl <- NULL
}


### Simple and Multiple Linear Regression with Parallel Bootstrapping

Bootstrapping estimates variability of coefficients by resampling data with replacement. This is embarrassingly parallel; we use {snow} with `parLapplyLB` for bootstrap iterations.


In [ ]:
%%R
library(snow)
n_cores <- parallel::detectCores() - 1
cl <- NULL
tryCatch({
  cl <- makeCluster(n_cores, type = "SOCK")
}, error = function(e) {
  cat("Error creating cluster:", conditionMessage(e), "\n")
  stop("Cluster creation failed.")
})
vars <- c("trip_distance", "fare_amount", "passenger_count", "trip_duration")
reg_df <- df_processed[, vars, drop = FALSE]
reg_df <- reg_df[complete.cases(reg_df) & apply(reg_df, 1, function(x) all(is.finite(x))), ]
bootstrap_lm <- function(data, idx, formula) {
  model <- lm(formula, data = data[idx, ])
  coef(model)
}
n_boot <- 1000
boot_indices <- lapply(1:n_boot, function(i) sample(1:nrow(reg_df), replace = TRUE))


#### Simple Linear Regression (SLR) — bootstrap


In [ ]:
%%R
ols_model <- lm(fare_amount ~ trip_distance, data = reg_df)
cat("OLS Coefficients:\n")
print(summary(ols_model)$coefficients)
ols_formula <- fare_amount ~ trip_distance
clusterExport(cl, c("bootstrap_lm", "reg_df", "ols_formula"))
boot_results_ols <- parLapplyLB(cl, boot_indices, function(idx) bootstrap_lm(reg_df, idx, ols_formula))
boot_coefs_ols <- do.call(rbind, boot_results_ols)
ols_ci <- apply(boot_coefs_ols, 2, quantile, probs = c(0.025, 0.975))
cat("\nOLS Bootstrap 95% CIs:\n")
print(round(ols_ci, 3))


In [ ]:
%%R
ols_boot_df <- as.data.frame(boot_coefs_ols)
colnames(ols_boot_df) <- c("(Intercept)", "trip_distance")
ols_seq_coefs <- coef(ols_model)
par(mfrow = c(1, 2))
for (coef_name in colnames(ols_boot_df)) {
  hist(ols_boot_df[[coef_name]], main = paste("Bootstrap Distribution:", coef_name),
       xlab = "Coefficient Estimate", col = "lightblue", breaks = 30)
  abline(v = ols_seq_coefs[coef_name], col = "red", lwd = 2, lty = 2)
}
par(mfrow = c(1, 1))


#### Multiple Linear Regression (MLR) — bootstrap


In [ ]:
%%R
mlr_model <- lm(fare_amount ~ trip_distance + passenger_count + trip_duration, data = reg_df)
cat("\nMLR Coefficients:\n")
print(summary(mlr_model)$coefficients)
mlr_formula <- fare_amount ~ trip_distance + passenger_count + trip_duration
clusterExport(cl, c("bootstrap_lm", "reg_df", "mlr_formula"))
boot_results_mlr <- parLapplyLB(cl, boot_indices, function(idx) bootstrap_lm(reg_df, idx, mlr_formula))
boot_coefs_mlr <- do.call(rbind, boot_results_mlr)
mlr_ci <- apply(boot_coefs_mlr, 2, quantile, probs = c(0.025, 0.975))
cat("\nMLR Bootstrap 95% CIs:\n")
print(round(mlr_ci, 3))


In [ ]:
%%R
mlr_boot_df <- as.data.frame(boot_coefs_mlr)
colnames(mlr_boot_df) <- c("(Intercept)", "trip_distance", "passenger_count", "trip_duration")
mlr_seq_coefs <- coef(mlr_model)
par(mfrow = c(2, 2))
for (coef_name in colnames(mlr_boot_df)) {
  hist(mlr_boot_df[[coef_name]], main = paste("Bootstrap Distribution:", coef_name),
       xlab = "Coefficient Estimate", col = "lightgreen", breaks = 30)
  abline(v = mlr_seq_coefs[coef_name], col = "red", lwd = 2, lty = 2)
}
par(mfrow = c(1, 1))
mlr_boot_long <- reshape2::melt(mlr_boot_df, variable.name = "Coefficient", value.name = "Estimate")
ggplot(mlr_boot_long, aes(x = Estimate, fill = Coefficient)) +
  geom_density(alpha = 0.5) +
  geom_vline(data = data.frame(Coefficient = names(mlr_seq_coefs), Estimate = mlr_seq_coefs),
             aes(xintercept = Estimate), color = "red", linetype = "dashed") +
  facet_wrap(~ Coefficient, scales = "free") +
  theme_minimal() +
  labs(title = "MLR Bootstrap Coefficient Distributions", x = "Coefficient Estimate", y = "Density")


In [ ]:
%%R
if (!is.null(cl)) {
  tryCatch({
    stopCluster(cl)
  }, error = function(e) {
    cat("Error stopping cluster:", conditionMessage(e), "\n")
  })
  cl <- NULL
}


## Cleanup and Best Practices

- Always stop the cluster: `stopCluster(cl)`
- Monitor memory: For very large data, use smaller chunks or sample first.
- Compare timings: Use `system.time()` for sequential vs. parallel to see speedups.
- Error handling: Wrap functions in `tryCatch` for robustness.


## Summary and Conclusion

The {snow} package is a powerful tool for parallel computing in R, making it easier to speed up computations by distributing tasks across multiple cores or machines. Its simplicity and flexibility make it ideal for embarrassingly parallel problems, though users must consider overhead and setup requirements for optimal use. By leveraging functions like `parLapply` and `clusterApply`, users can significantly reduce computation time for large-scale analyses. This tutorial demonstrated how to set up a {snow} cluster, process large datasets in parallel, and perform statistical analyses efficiently. Always remember to stop the cluster after use to free up resources.


## References

1. **{snow} Package Documentation (CRAN)** — [CRAN: snow](https://cran.r-project.org/package=snow) — Tierney, L., et al. (2023). snow: Simple Network of Workstations. R package version 0.4-4.

2. **"Parallel R" by McCallum and Weston** — [O'Reilly](https://www.oreilly.com/library/view/parallel-r/9781449309923/) — McCallum, Q. E., & Weston, S. (2011). *Parallel R*. O'Reilly Media.

3. **"Applied Linear Statistical Models" by Kutner et al.** — Kutner, M. H., et al. (2005). *Applied Linear Statistical Models* (5th ed.). McGraw-Hill.
